# Colab Pro capability check — GIAB HG001 WES

Run these cells in your Colab Pro session. This inspects the allocated host and existing project Drive root. It does not build an index, install Docker, download genomic assets or start HG001 processing. The resulting report remains explicitly noncanonical.

**Owner storage requirement:** index construction will use Colab CPU/RAM and `/content` scratch. Durable large files belong beneath `giab-wes-nextflow-private`: verified sources in `cache/verified-sources/`, reference/index derivatives in `cache/reference-assets/sha256/`, validated outputs in `runs/`, records in `registry/runs/`. Nextflow work stays off Drive. No construction on the owner's Mac.

**Pinned code:** `676718ed6c02c3076a26b48fb82d684251fb7892`. This is the reviewed implementation commit; this notebook is committed separately. The notebook is a thin launcher; capability logic and provenance checks live in the installed package. No saved execution output is included.


In [ ]:
import sys
import subprocess
import tempfile
from pathlib import Path
from google.colab import drive, files

if sys.version_info < (3, 12):
    raise RuntimeError("This package requires Python 3.12 or newer; return the runtime version before proceeding.")
if not Path("/content").is_dir():
    raise RuntimeError("Run this notebook in the owner's Colab session.")
REPOSITORY = "https://github.com/jcollins-bioinfo/giab-wes-nextflow.git"
REPOSITORY_SHA = "676718ed6c02c3076a26b48fb82d684251fb7892"
STAGE = Path(tempfile.mkdtemp(prefix="giab-capability-", dir="/content"))
CHECKOUT = STAGE / "checkout"

def run(args, cwd=None):
    """Execute a checked launcher command without shell interpolation."""
    return subprocess.run(args, cwd=cwd, check=True, text=True, capture_output=True).stdout.strip()

run(["git", "clone", "--no-checkout", "--filter=blob:none", REPOSITORY, str(CHECKOUT)])
if run(["git", "remote", "get-url", "origin"], CHECKOUT) != REPOSITORY:
    raise RuntimeError("Unexpected repository origin")
run(["git", "fetch", "--depth=1", "origin", REPOSITORY_SHA], CHECKOUT)
run(["git", "checkout", "--detach", REPOSITORY_SHA], CHECKOUT)
if run(["git", "rev-parse", "HEAD"], CHECKOUT) != REPOSITORY_SHA:
    raise RuntimeError("Checkout identity mismatch")
if run(["git", "status", "--porcelain", "--untracked-files=normal"], CHECKOUT):
    raise RuntimeError("Checkout must be clean")
subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall", str(CHECKOUT)], check=True)
print(run([sys.executable, "-I", "-m", "giab_wes_nextflow.runtime_identity", "--source-root", str(CHECKOUT), "--expected-sha", REPOSITORY_SHA]))


## Mount the existing project folder and inspect capabilities

Authorize the native Drive mount prompt. Only the named project root is checked; the launcher does not browse your other folders. Filesystem-reported free space does not establish Drive account quota. The index memory estimate is documentation-derived, before headroom, not measured RSS or a canonical readiness decision.


In [ ]:
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/giab-wes-nextflow-private")
REPORT = STAGE / "canonical-host-capability.json"
print(run([sys.executable, "-I", "-m", "giab_wes_nextflow.canonical_host",
    "--source-root", str(CHECKOUT), "--expected-sha", REPOSITORY_SHA,
    "--drive-root", str(DRIVE_ROOT), "--scratch", "/content", "--output", str(REPORT)]))


## Return the report

Download the small JSON report and attach it to the project conversation. It includes the verified code identity and selected host metadata, not private directory listings or sequence data. If a cell fails, return the error instead of continuing to index construction. Review memory, scratch disk and pinned-runtime qualification before any expensive job.


In [ ]:
files.download(str(REPORT))
